# ASVspoof 5 Track 1 - WavLM-Base

WavLM-Base(frozen) + adapter + classifier 2x2 조합을 train 50% subset으로 학습하고, best.pt(dev 5,000개 minDCF 기준)로 Track 1 eval 680,774개 전체를 점수화한 뒤 주최측 evaluation-package로 채점한다.

- A1C1 BiLSTM + LCNN-big / A1C2 BiLSTM + SENet12 / A2C1 SE-Res2Net + LCNN-big / A2C2 SE-Res2Net + SENet12
- MUSAN 미사용 (eval plan 4.2), RawBoost / codec / RIR / SpecAugment는 baseline.yaml 설정 그대로
- 음성은 `/content`에 받고 체크포인트, 점수, 로그는 Drive의 `experiments/<이름>/`에 저장

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
PROJECT = '/content/drive/MyDrive/Project/ASVspoof5'   # repo를 둔 경로
EXPERIMENT = 'wavlm_base_no_musan_20261106'
OUTPUT = f'{PROJECT}/experiments/{EXPERIMENT}'
%cd {PROJECT}
!pip install -q -r requirements.txt

## 학습과 dev / eval 점수

데이터 다운로드, 50% subset 선택, smoke test, 조합별 학습, 전체 dev/eval 점수 생성까지 한 번에 돈다. 끝난 조합은 건너뛰고 `last.pt`가 있으면 이어서 학습한다.

In [ ]:
!python scripts/run_wavlm_base.py --project {PROJECT} --experiment {EXPERIMENT} --combos a1c1 a1c2 a2c1 a2c2

## 공식 채점

주최측 repo(commit `fe23d30`)의 `evaluation-package/evaluation.py`로 minDCF / EER / actDCF / Cllr를 다시 계산한다. 점수 개수, ID 중복, 라벨이 프로토콜과 맞는지도 같이 확인한다.

In [ ]:
!git clone -q https://github.com/asvspoof-challenge/asvspoof5.git /content/asvspoof5_official
!git -C /content/asvspoof5_official checkout -q fe23d3053a0889b305d98c9e4cadf418c213d972
!python scripts/official_track1_report.py --output {OUTPUT}

## 결과

In [ ]:
import json, glob, pandas as pd
rows = [json.load(open(p)) for p in sorted(glob.glob(f'{OUTPUT}/reports/official_eval_*.json'))]
pd.DataFrame([{k: r[k] for k in ['combo', 'architecture', 'rows', 'min_dcf', 'eer_pct', 'act_dcf', 'cllr']} for r in rows])